# 01 - Data Download & Exploratory Data Analysis
**IS5126 Cross-Regional PD Model Transferability Study**

This notebook covers:
1. Download Lending Club dataset from Kaggle
2. Data overview & quality assessment
3. Target variable (default) definition
4. Feature exploration & distributions
5. Missing value analysis
6. Correlation & multicollinearity check
7. Export cleaned dataset for next steps

## 0. Setup & Install

In [ ]:
# Run this cell first in Colab
!pip install -q kagglehub shap category_encoders lightgbm xgboost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.3f}'.format)

# Plot style
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

# Color palette for the project
COLORS = {
    'primary': '#1F4E79',
    'accent': '#2E86AB',
    'good': '#28A745',
    'bad': '#DC3545',
    'neutral': '#6C757D',
    'light': '#D6E4F0'
}

print('Setup complete!')

## 1. Download Lending Club Data

We use the Lending Club dataset from Kaggle. This is one of the most widely used datasets for credit risk modeling.

**Dataset:** https://www.kaggle.com/datasets/wordsforthewise/lending-club

In [ ]:
import kagglehub

# Download the dataset (will prompt for Kaggle credentials on first run)
# In Colab: you'll need to upload your kaggle.json or authenticate
path = kagglehub.dataset_download("wordsforthewise/lending-club")
print(f"Dataset downloaded to: {path}")

In [ ]:
# List files in download directory
import os
for f in os.listdir(path):
    size_mb = os.path.getsize(os.path.join(path, f)) / 1024 / 1024
    print(f"{f}: {size_mb:.1f} MB")

In [ ]:
# Load accepted loans data
# This is the main dataset - rejected loans don't have outcome labels
csv_path = os.path.join(path, 'accepted_2007_to_2018Q4.csv')

# If the file name is different, adjust accordingly:
# csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
# print(csv_files)  # uncomment to check

print("Loading data... (this may take a minute)")
df_raw = pd.read_csv(csv_path, low_memory=False)
print(f"Loaded: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

## 2. Data Overview

In [ ]:
# First look at the data
df_raw.head(3)

In [ ]:
# Data types overview
dtype_summary = pd.DataFrame({
    'dtype': df_raw.dtypes,
    'non_null': df_raw.notnull().sum(),
    'null_count': df_raw.isnull().sum(),
    'null_pct': (df_raw.isnull().sum() / len(df_raw) * 100).round(2),
    'nunique': df_raw.nunique(),
    'sample_value': df_raw.iloc[0]
})
dtype_summary.sort_values('null_pct', ascending=False).head(30)

In [ ]:
# Quick stats on numeric columns
df_raw.describe().T.head(20)

## 3. Target Variable Definition

This is a **critical step** in credit risk modeling. We need to clearly define what "default" means.

In Lending Club data, `loan_status` is the key field. We map it to a binary target:
- **Default (1):** Charged Off, Default, Late (31-120 days)
- **Non-default (0):** Fully Paid
- **Excluded:** Current, In Grace Period, Late (16-30 days) — outcome not yet determined

In [ ]:
# Check all loan_status values
print("=== loan_status Distribution ===")
status_counts = df_raw['loan_status'].value_counts()
status_pct = df_raw['loan_status'].value_counts(normalize=True) * 100

status_df = pd.DataFrame({'Count': status_counts, 'Percentage': status_pct.round(2)})
print(status_df)
print(f"\nTotal: {len(df_raw):,}")

In [ ]:
# Define default mapping
default_statuses = ['Charged Off', 'Default', 'Late (31-120 days)']
non_default_statuses = ['Fully Paid']
excluded_statuses = ['Current', 'In Grace Period', 'Late (16-30 days)',
                     'Does not meet the credit policy. Status:Fully Paid',
                     'Does not meet the credit policy. Status:Charged Off']

# Filter to loans with determined outcomes only
df = df_raw[df_raw['loan_status'].isin(default_statuses + non_default_statuses)].copy()

# Create binary target
df['default'] = df['loan_status'].isin(default_statuses).astype(int)

print(f"After filtering: {len(df):,} loans ({len(df)/len(df_raw)*100:.1f}% of original)")
print(f"\nTarget distribution:")
print(df['default'].value_counts())
print(f"\nDefault rate: {df['default'].mean()*100:.2f}%")

In [ ]:
# Visualize target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts = df['default'].value_counts()
bars = axes[0].bar(['Non-Default (0)', 'Default (1)'], counts.values,
                    color=[COLORS['good'], COLORS['bad']], edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
                 f'{val:,}', ha='center', fontweight='bold', fontsize=12)
axes[0].set_title('Target Variable Distribution', fontweight='bold')
axes[0].set_ylabel('Count')

# Default rate over time
if 'issue_d' in df.columns:
    df['issue_date'] = pd.to_datetime(df['issue_d'], format='mixed', errors='coerce')
    monthly_default = df.groupby(df['issue_date'].dt.to_period('Q'))['default'].mean() * 100
    monthly_default.plot(ax=axes[1], color=COLORS['primary'], linewidth=2)
    axes[1].set_title('Default Rate Over Time (Quarterly)', fontweight='bold')
    axes[1].set_ylabel('Default Rate (%)')
    axes[1].set_xlabel('Issue Quarter')
    axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('figures/01_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n⚠️ Note: Class imbalance is expected in credit risk. ~20% default rate is typical for Lending Club.")
print("We'll handle this with stratified sampling and appropriate metrics (not just accuracy).")

## 4. Remove Data Leakage Features

**Critical for credit risk modeling:** Some features are only available *after* the loan has been issued and cannot be used for prediction at origination time. Using them would cause data leakage.

We also remove features with excessive missing values or no predictive value.

In [ ]:
# Features to drop - grouped by reason

# Post-origination / leakage features (available only after loan is issued)
leakage_cols = [
    'funded_amnt', 'funded_amnt_inv',          # funding info (post-listing)
    'total_pymnt', 'total_pymnt_inv',          # payment info (post-origination)
    'total_rec_prncp', 'total_rec_int',        # received principal/interest
    'total_rec_late_fee',                       # late fees collected
    'recoveries', 'collection_recovery_fee',   # post-default recovery
    'last_pymnt_d', 'last_pymnt_amnt',         # last payment info
    'next_pymnt_d',                            # next payment date
    'last_credit_pull_d',                      # LC's credit pull
    'out_prncp', 'out_prncp_inv',              # outstanding principal
    'hardship_flag', 'hardship_type',          # hardship program
    'hardship_reason', 'hardship_status',
    'hardship_start_date', 'hardship_end_date',
    'hardship_amount', 'hardship_length',
    'hardship_dpd', 'hardship_loan_status',
    'hardship_payoff_balance_amount',
    'hardship_last_payment_amount',
    'payment_plan_start_date',
    'debt_settlement_flag',
    'debt_settlement_flag_date',
    'settlement_status', 'settlement_date',
    'settlement_amount', 'settlement_percentage',
    'settlement_term',
]

# Identifiers / non-predictive
id_cols = ['id', 'member_id', 'url', 'policy_code']

# Redundant / will derive better features
redundant_cols = ['loan_status', 'issue_d', 'issue_date',
                  'zip_code',  # too granular, use addr_state instead
                  'title',     # mostly duplicates 'purpose'
]

# Combine all columns to drop
drop_cols = leakage_cols + id_cols + redundant_cols

# Only drop columns that actually exist in the dataframe
existing_drop = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=existing_drop)

print(f"Dropped {len(existing_drop)} columns")
print(f"Remaining: {df.shape[1]} columns")

In [ ]:
# Drop columns with > 70% missing values
missing_pct = df.isnull().mean()
high_missing = missing_pct[missing_pct > 0.70].index.tolist()

print(f"Columns with >70% missing ({len(high_missing)}):")
for col in high_missing:
    print(f"  {col}: {missing_pct[col]*100:.1f}% missing")

df = df.drop(columns=high_missing)
print(f"\nAfter dropping high-missing columns: {df.shape[1]} columns remain")

## 5. Feature Categorization

Organize remaining features by business meaning. This is essential for understanding what we're modeling.

In [ ]:
# Categorize features by business domain
feature_categories = {
    'Loan Characteristics': [
        'loan_amnt', 'term', 'int_rate', 'installment',
        'grade', 'sub_grade', 'purpose',
    ],
    'Borrower Profile': [
        'emp_title', 'emp_length', 'home_ownership',
        'annual_inc', 'verification_status', 'addr_state',
    ],
    'Credit History': [
        'dti', 'delinq_2yrs', 'earliest_cr_line',
        'fico_range_low', 'fico_range_high',
        'inq_last_6mths', 'open_acc', 'pub_rec',
        'revol_bal', 'revol_util', 'total_acc',
        'initial_list_status', 'application_type',
    ],
    'Extended Credit Metrics': [
        'mort_acc', 'pub_rec_bankruptcies',
        'num_accts_ever_120_pd', 'num_actv_bc_tl',
        'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl',
        'num_il_tl', 'num_op_rev_tl', 'num_rev_accts',
        'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_30dpd',
        'num_tl_90g_dpd_24m', 'num_tl_op_past_12m',
        'pct_tl_nvr_dlq', 'percent_bc_gt_75',
        'tot_coll_amt', 'tot_cur_bal', 'tot_hi_cred_lim',
        'total_bal_ex_mort', 'total_bc_limit',
        'total_il_high_credit_limit',
        'acc_open_past_24mths', 'avg_cur_bal',
        'bc_open_to_buy', 'bc_util',
        'chargeoff_within_12_mths', 'collections_12_mths_ex_med',
        'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op',
        'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl',
        'mths_since_recent_bc', 'mths_since_recent_inq',
        'tax_liens', 'total_bal_il', 'total_cu_tl',
        'total_rev_hi_lim',
    ],
    'Text Features (for LLM)': [
        'emp_title', 'desc',
    ]
}

# Print categorized features
for category, features in feature_categories.items():
    existing = [f for f in features if f in df.columns]
    print(f"\n{'='*50}")
    print(f"{category} ({len(existing)} features)")
    print('='*50)
    for f in existing:
        dtype = df[f].dtype
        null_pct = df[f].isnull().mean() * 100
        if dtype in ['float64', 'int64']:
            print(f"  {f:35s} | {str(dtype):8s} | {null_pct:5.1f}% null | mean={df[f].mean():.2f}")
        else:
            print(f"  {f:35s} | {str(dtype):8s} | {null_pct:5.1f}% null | {df[f].nunique()} unique")

## 6. Key Feature Distributions by Default Status

This is the most important part of EDA for credit risk: understanding how features differ between defaulters and non-defaulters.

In [ ]:
# Key numeric features to analyze
key_numeric = ['loan_amnt', 'int_rate', 'annual_inc', 'dti',
               'fico_range_low', 'revol_util', 'revol_bal',
               'installment', 'open_acc', 'total_acc',
               'inq_last_6mths', 'delinq_2yrs']

# Only keep features that exist in our dataframe
key_numeric = [f for f in key_numeric if f in df.columns]

# Clean int_rate if it's stored as string (e.g., "13.56%")
if df['int_rate'].dtype == 'object':
    df['int_rate'] = df['int_rate'].str.rstrip('%').astype(float)

# Clean revol_util similarly
if 'revol_util' in df.columns and df['revol_util'].dtype == 'object':
    df['revol_util'] = df['revol_util'].str.rstrip('%').astype(float)

# Clean term
if df['term'].dtype == 'object':
    df['term'] = df['term'].str.strip().str.replace(' months', '').astype(int)

In [ ]:
# Distribution plots: defaulters vs non-defaulters
n_features = len(key_numeric)
n_cols = 3
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(key_numeric):
    ax = axes[i]
    # Remove extreme outliers for visualization (cap at 99th percentile)
    upper = df[col].quantile(0.99)
    plot_data = df[df[col] <= upper] if pd.notna(upper) else df

    for label, color in [(0, COLORS['good']), (1, COLORS['bad'])]:
        subset = plot_data[plot_data['default'] == label][col].dropna()
        ax.hist(subset, bins=50, alpha=0.5, color=color, density=True,
                label='Non-Default' if label == 0 else 'Default')

    ax.set_title(col, fontweight='bold')
    ax.legend(fontsize=8)

# Hide empty subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions by Default Status', fontweight='bold', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('figures/02_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Statistical comparison: mean values by default status
comparison = df.groupby('default')[key_numeric].mean().T
comparison.columns = ['Non-Default', 'Default']
comparison['Diff (%)'] = ((comparison['Default'] - comparison['Non-Default']) / comparison['Non-Default'] * 100).round(1)
comparison = comparison.round(3)

print("=== Mean Feature Values by Default Status ===")
print("(Positive Diff% = higher for defaulters)\n")
print(comparison.to_string())

## 7. Categorical Feature Analysis

In [ ]:
# Key categorical features
key_categorical = ['grade', 'sub_grade', 'home_ownership', 'purpose',
                   'verification_status', 'term', 'application_type']
key_categorical = [f for f in key_categorical if f in df.columns]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(['grade', 'home_ownership', 'purpose', 'verification_status']):
    if col not in df.columns:
        continue
    ax = axes[i]

    # Calculate default rate per category
    cat_default = df.groupby(col)['default'].agg(['mean', 'count'])
    cat_default.columns = ['default_rate', 'count']
    cat_default = cat_default.sort_values('default_rate', ascending=True)

    # Filter to categories with meaningful sample size
    cat_default = cat_default[cat_default['count'] >= 100]

    bars = ax.barh(range(len(cat_default)), cat_default['default_rate'] * 100,
                   color=COLORS['accent'], edgecolor='white')
    ax.set_yticks(range(len(cat_default)))
    ax.set_yticklabels(cat_default.index, fontsize=9)
    ax.set_xlabel('Default Rate (%)')
    ax.set_title(f'Default Rate by {col}', fontweight='bold')

    # Add count labels
    for j, (rate, count) in enumerate(zip(cat_default['default_rate'], cat_default['count'])):
        ax.text(rate * 100 + 0.3, j, f'n={count:,}', va='center', fontsize=8, color=COLORS['neutral'])

plt.tight_layout()
plt.savefig('figures/03_categorical_default_rates.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Grade analysis - this is critical for credit risk understanding
grade_analysis = df.groupby('grade').agg(
    count=('default', 'count'),
    default_rate=('default', 'mean'),
    avg_int_rate=('int_rate', 'mean'),
    avg_loan_amt=('loan_amnt', 'mean'),
    avg_income=('annual_inc', 'mean'),
    avg_dti=('dti', 'mean')
).round(3)

print("=== Loan Grade Analysis ===")
print("(Grade A = lowest risk, Grade G = highest risk)\n")
print(grade_analysis.to_string())

## 8. Missing Value Analysis

In [ ]:
# Missing value heatmap for features with any missing data
missing = df.isnull().mean()
missing_features = missing[missing > 0].sort_values(ascending=False)

print(f"Features with missing values: {len(missing_features)} out of {df.shape[1]}\n")

if len(missing_features) > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, len(missing_features) * 0.3)))
    bars = ax.barh(range(len(missing_features)), missing_features.values * 100,
                   color=COLORS['accent'])
    ax.set_yticks(range(len(missing_features)))
    ax.set_yticklabels(missing_features.index, fontsize=9)
    ax.set_xlabel('Missing %')
    ax.set_title('Missing Values by Feature', fontweight='bold')
    ax.invert_yaxis()

    for i, val in enumerate(missing_features.values):
        ax.text(val * 100 + 0.3, i, f'{val*100:.1f}%', va='center', fontsize=8)

    plt.tight_layout()
    plt.savefig('figures/04_missing_values.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Check if missingness is related to default (MAR vs MCAR)
# This is important for choosing imputation strategy
print("=== Missing Rate by Default Status ===")
print("(Large differences suggest Missing At Random - MAR)\n")

for col in missing_features.head(10).index:
    miss_default = df[df['default'] == 1][col].isnull().mean() * 100
    miss_non = df[df['default'] == 0][col].isnull().mean() * 100
    diff = abs(miss_default - miss_non)
    flag = " ⚠️" if diff > 2 else ""
    print(f"  {col:35s} | Non-Default: {miss_non:5.1f}% | Default: {miss_default:5.1f}% | Diff: {diff:.1f}%{flag}")

## 9. Correlation Analysis

In [ ]:
# Correlation with target variable
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'default']

target_corr = df[numeric_cols + ['default']].corr()['default'].drop('default').sort_values()

# Plot top and bottom correlations
fig, ax = plt.subplots(figsize=(10, 8))
top_bottom = pd.concat([target_corr.head(15), target_corr.tail(15)])
colors = [COLORS['bad'] if v > 0 else COLORS['good'] for v in top_bottom.values]
ax.barh(range(len(top_bottom)), top_bottom.values, color=colors)
ax.set_yticks(range(len(top_bottom)))
ax.set_yticklabels(top_bottom.index, fontsize=9)
ax.set_xlabel('Correlation with Default')
ax.set_title('Top Features Correlated with Default', fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.savefig('figures/05_target_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Multicollinearity check - feature-to-feature correlation
# Focus on highly correlated pairs (potential redundancy)
corr_matrix = df[numeric_cols].corr()

# Find pairs with |correlation| > 0.8
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.8:
            high_corr_pairs.append({
                'Feature 1': corr_matrix.columns[i],
                'Feature 2': corr_matrix.columns[j],
                'Correlation': round(corr_matrix.iloc[i, j], 3)
            })

if high_corr_pairs:
    corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation', ascending=False)
    print(f"=== Highly Correlated Feature Pairs (|r| > 0.8) ===")
    print(f"Total: {len(corr_df)} pairs\n")
    print(corr_df.to_string(index=False))
    print("\n⚠️ Consider dropping one from each pair or using PCA/VIF analysis.")
else:
    print("No highly correlated pairs found (|r| > 0.8).")

## 10. Text Features Preview (for LLM Processing)

Quick look at `emp_title` and `desc` — these will be processed with LLM in notebook 02.

In [ ]:
# emp_title analysis
if 'emp_title' in df.columns:
    print(f"emp_title: {df['emp_title'].nunique():,} unique values")
    print(f"Missing: {df['emp_title'].isnull().mean()*100:.1f}%")
    print(f"\nTop 20 most common:")
    print(df['emp_title'].value_counts().head(20))

    # Show the messiness - many variations of similar jobs
    print("\n\n--- Examples of messy job titles (why we need LLM) ---")
    teacher_variants = df[df['emp_title'].str.contains('teach', case=False, na=False)]['emp_title'].value_counts().head(10)
    print(f"\nVariations containing 'teach':")
    print(teacher_variants)

In [ ]:
# desc field analysis (borrower's loan description)
if 'desc' in df.columns:
    print(f"desc: {df['desc'].notnull().sum():,} non-null ({df['desc'].notnull().mean()*100:.1f}%)")
    print(f"\nSample descriptions:")
    for desc in df['desc'].dropna().head(5):
        print(f"  - {str(desc)[:120]}...")
else:
    print("'desc' column not found in this version of the dataset.")
    print("No problem - we can still use emp_title for LLM features.")

## 11. Save Processed Data

In [ ]:
# Create output directory
os.makedirs('data/processed', exist_ok=True)
os.makedirs('figures', exist_ok=True)

# Save cleaned dataframe
output_path = 'data/processed/lending_club_cleaned.parquet'
df.to_parquet(output_path, index=False)

print(f"Saved cleaned data to: {output_path}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Default rate: {df['default'].mean()*100:.2f}%")
print(f"\nFile size: {os.path.getsize(output_path)/1024/1024:.1f} MB")

In [ ]:
# Save emp_title separately for LLM processing (smaller file)
if 'emp_title' in df.columns:
    emp_titles = df[['emp_title', 'default']].dropna(subset=['emp_title'])
    emp_titles.to_csv('data/processed/emp_titles_for_llm.csv', index=False)
    print(f"Saved {len(emp_titles):,} emp_titles for LLM processing")

## 12. EDA Summary & Key Findings

Document your observations here after running the notebook. These findings will guide feature engineering in notebook 02.

**Template (fill in after running):**

- Dataset size: ___ loans after filtering
- Default rate: ___% (class imbalance level: moderate/severe)
- Strongest predictors: ___, ___, ___
- Key multicollinearity issues: ___
- Missing value strategy needed for: ___
- emp_title has ___ unique values → LLM classification needed
- Features that will be most affected in cross-regional transfer: ___

---

**Next steps → Notebook 02: Feature Engineering**